In [1]:
using Distributed
using Dates

# --- 1. SETUP WORKERS (Notebook Safe) ---
# This check prevents "adding workers" forever if you re-run the cell
if nprocs() == 1
    addprocs(4) 
end

# --- 2. LOAD PACKAGES ON ALL WORKERS ---
@everywhere begin
    using GenX
    using Dates
    using Gurobi
end

# --- 3. DEFINE THE RUN LOGIC ---
@everywhere function process_scenario(job_data)
    (demand_file, var_file, run_name, base_settings_path, demand_lib_path, var_file_full_path, output_root) = job_data

    run_dir = joinpath(output_root, run_name)
    
    # 1. SKIP logic: If the folder is there, keep it. 
    if isdir(run_dir)
        return "SKIPPED: $run_name already exists."
    end

    try
        # 2. CREATE AND MIRROR EVERYTHING
        # We copy the ENTIRE Base_Settings folder tree to Run_Dir in one step.
        # This includes all subfolders (inputs, settings, policies, etc.)
        cp(base_settings_path, run_dir)

        # 3. SET TARGET PATH
        # We know exactly where the dynamic files go within that structure:
        target_path = joinpath(run_dir, "system")
        
        # Verify the mirrored structure has the expected landing zone
        if !isdir(target_path)
            mkpath(target_path) 
        end

        # 3. Rename while copying: library file -> standardized model name
        cp(joinpath(demand_lib_path, demand_file), joinpath(target_path, "Demand_data.csv"), force=true)

        # 4. Rename while copying: variability library file -> standardized model name
        cp(var_file_full_path, joinpath(target_path, "Generators_variability.csv"), force=true)
        
        # 5. EXECUTION
        # Change directory into the run_dir so GenX sees everything as local
        cd(run_dir) do
            run_genx_case!(pwd(), Gurobi.Optimizer) 
        end
        
        return "SUCCESS: $run_name"

    catch e
        return "ERROR: $run_name failed with $e"
    end
end

# --- 4. MAIN CONFIGURATION ---
base_settings_path = "Base_Settings" 
demand_lib_path    = "Library_Demand"
var_lib_root       = "Library_Variability"
output_root        = "Scenarios"

if !isdir(output_root)
    mkdir(output_root)
end

# --- 5. BUILD JOB LIST ---
jobs = []
all_demands = filter(x -> endswith(x, ".csv"), readdir(demand_lib_path))

for d_file in all_demands
    stem = replace(d_file, ".csv" => "")
    specific_var_folder = joinpath(var_lib_root, stem)
    
    if isdir(specific_var_folder)
        var_files = filter(x -> endswith(x, ".csv"), readdir(specific_var_folder))
        for v_file in var_files
            run_id = replace(v_file, ".csv" => "") 
            run_name = "Run_$run_id"
            var_full_path = joinpath(specific_var_folder, v_file)
            
            push!(jobs, (d_file, v_file, run_name, base_settings_path, demand_lib_path, var_full_path, output_root))
        end
    end
end
test_jobs = jobs[1:1]

println("Prepared $(length(jobs)) scenarios.")

# --- 6. EXECUTE ---
results = pmap(process_scenario, jobs)

# --- 6. REPORTING ---
for r in results
    println(r)
end

Prepared 400 scenarios.
      From worker 4:	  ____           __  __   _ _
      From worker 4:	 / ___| ___ _ __ \ \/ /  (_) |
      From worker 4:	| |  _ / _ \ '_ \ \  /   | | |
      From worker 4:	| |_| |  __/ | | |/  \ _ | | |
      From worker 4:	 \____|\___|_| |_/_/\_(_)/ |_|
      From worker 4:	                       |__/
      From worker 4:	 Version: 0.4.5
      From worker 3:	  ____           __  __   _ _
      From worker 3:	 / ___| ___ _ __ \ \/ /  (_) |
      From worker 3:	| |  _ / _ \ '_ \ \  /   | | |
      From worker 3:	| |_| |  __/ | | |/  \ _ | | |
      From worker 3:	 \____|\___|_| |_/_/\_(_)/ |_|
      From worker 3:	                       |__/
      From worker 3:	 Version: 0.4.5
      From worker 3:	
      From worker 3:	Configuring Settings
      From worker 5:	  ____           __  __   _ _
      From worker 5:	 / ___| ___ _ __ \ \/ /  (_) |
      From worker 5:	| |  _ / _ \ '_ \ \  /   | | |
      From worker 5:	| |_| |  __/ | | |/  \ _ | | |
      From work

Excessive output truncated after 524305 bytes.